# AI-Based Delivery Delay Risk Dataset Generation
This notebook creates a synthetic dataset for modeling multi-class delivery delay risk in e-commerce logistics.

In [13]:
# ============================================================
# AI-BASED DELIVERY DELAY RISK DATASET GENERATION
# ============================================================
# This notebook generates a realistic synthetic dataset for
# predicting delivery delay risk levels in e-commerce logistics.
# ============================================================

!pip install pandas  # Uncomment if pandas is not installed
import numpy as np
import pandas as pd


In [14]:
# Set random seed for reproducibility
# Ensures the same dataset can be regenerated if needed
np.random.seed(42)


In [15]:

# ------------------------------------------------------------
# STEP 1: DEFINE DATASET SIZE
# ------------------------------------------------------------
# Total number of delivery records (orders)
n = 5000


In [16]:

# ------------------------------------------------------------
# STEP 2: GENERATE RAW FEATURES
# ------------------------------------------------------------
# These features are inspired by real-world supply chain and
# logistics operations.

data = pd.DataFrame({
    # Number of orders handled by the warehouse on that day
    # Higher volume increases processing and dispatch delays
    "order_volume": np.random.randint(10, 500, n),

    # Time taken by the warehouse to process the order (in hours)
    # Includes picking, packing, and dispatch preparation
    "warehouse_time_hrs": np.random.uniform(1, 48, n),

    # Distance between warehouse and customer delivery location (in km)
    "shipment_distance_km": np.random.uniform(5, 3000, n),

    # Traffic congestion level on delivery route
    # 1 = Low traffic, 5 = Severe congestion
    "traffic_level": np.random.randint(1, 6, n),

    # Weather impact severity
    # 0 = Clear, 1 = Mild, 2 = Moderate, 3 = Severe
    "weather_severity": np.random.randint(0, 4, n),

    # Utilization level of delivery personnel (in percentage)
    # Higher load indicates stretched delivery capacity
    "courier_load_pct": np.random.uniform(30, 100, n),

    # Historical delay rate for the delivery route
    # Represents past reliability of the route
    "past_delay_rate": np.random.uniform(0.0, 0.6, n)
})


In [17]:

# ------------------------------------------------------------
# STEP 3: NORMALIZE FEATURES
# ------------------------------------------------------------
# Normalization ensures that features with larger numeric
# ranges do not dominate the risk score unfairly.

data["order_volume_norm"] = data["order_volume"] / data["order_volume"].max()
data["warehouse_time_norm"] = data["warehouse_time_hrs"] / data["warehouse_time_hrs"].max()
data["distance_norm"] = data["shipment_distance_km"] / data["shipment_distance_km"].max()
data["traffic_norm"] = data["traffic_level"] / 5
data["weather_norm"] = data["weather_severity"] / 3
data["courier_load_norm"] = data["courier_load_pct"] / 100
data["past_delay_norm"] = data["past_delay_rate"] / 0.6


In [18]:

# ------------------------------------------------------------
# STEP 4: COMPUTE DELIVERY RISK SCORE
# ------------------------------------------------------------
# A weighted risk score is calculated using domain intuition:
# - Traffic and weather have the strongest impact
# - Historical delays strongly influence future outcomes
# - Distance, warehouse time, and courier load have moderate impact
# - Order volume has relatively lower influence

data["risk_score"] = (
    0.10 * data["order_volume_norm"] +
    0.15 * data["warehouse_time_norm"] +
    0.15 * data["distance_norm"] +
    0.20 * data["traffic_norm"] +
    0.15 * data["weather_norm"] +
    0.10 * data["courier_load_norm"] +
    0.15 * data["past_delay_norm"]
)


In [19]:

# ------------------------------------------------------------
# STEP 5: ADD CONTROLLED STOCHASTIC NOISE
# ------------------------------------------------------------
# Real-world logistics are affected by unpredictable factors
# such as sudden traffic jams, minor accidents, or system delays.
# Small noise is added to simulate this uncertainty.

noise = np.random.uniform(-0.05, 0.05, n)
data["risk_score"] = data["risk_score"] + noise

# Ensure risk score remains within valid bounds
data["risk_score"] = data["risk_score"].clip(0, 1)


In [20]:

# ------------------------------------------------------------
# STEP 6: ASSIGN MULTI-CLASS DELIVERY STATUS
# ------------------------------------------------------------
# Instead of hard thresholds, probabilistic boundaries are used
# near class edges to avoid deterministic labeling.

def assign_delivery_status(risk):
    if risk < 0.45:
        return "On-Time"
    elif 0.45 <= risk <= 0.55:
        return np.random.choice(["On-Time", "At Risk"], p=[0.6, 0.4])
    elif 0.55 < risk < 0.70:
        return "At Risk"
    elif 0.70 <= risk <= 0.80:
        return np.random.choice(["At Risk", "Delayed"], p=[0.7, 0.3])
    else:
        return "Delayed"

# Apply classification logic
data["delivery_status"] = data["risk_score"].apply(assign_delivery_status)


In [21]:

# ------------------------------------------------------------
# STEP 7: CLEANUP INTERMEDIATE COLUMNS
# ------------------------------------------------------------
# Remove normalized helper columns to keep the dataset clean
# and realistic for downstream machine learning tasks.

data.drop(
    columns=[
        "order_volume_norm",
        "warehouse_time_norm",
        "distance_norm",
        "traffic_norm",
        "weather_norm",
        "courier_load_norm",
        "past_delay_norm"
    ],
    inplace=True
)


In [22]:

# ------------------------------------------------------------
# STEP 8: QUICK DATASET VALIDATION
# ------------------------------------------------------------

# View sample records
data.head()

# Check class distribution (should be naturally imbalanced)
data["delivery_status"].value_counts()

# Confirm dataset dimensions
data.shape


(5000, 9)

In [23]:

# ------------------------------------------------------------
# OPTIONAL: SAVE DATASET TO CSV
# ------------------------------------------------------------
# Uncomment if you want to persist the dataset
data.to_csv("delivery_delay_dataset.csv", index=False)

In [24]:
from google.colab import files
files.download("delivery_delay_dataset.csv")

ModuleNotFoundError: No module named 'google'